# Financial controls and next-period ROA

Adapted from the supplied original workflow. Outputs and personal paths have been removed. Read `docs/PREPARING_DATA.md` and configure `configs/preparation.local.json` first. Run cells in order. This adaptation has not been validated by a complete raw-to-neural rerun.


# Compustat financial preparation

Prepare an authorized local extract using the original financial rules. Run the configuration, function definitions, build, and verification cells in order. No raw financial records are distributed here.


## Step 1 — Imports and paths

Use your existing Python environment. If needed, uncomment `%pip install numpy pandas` once and restart the kernel. No package updates are needed in other running notebooks. Only one copy of this financial notebook should write to this output folder at a time.

RAW_SOURCE=None searches for the known original filename in the configured folders. It never guesses between downloads with different bytes. You can instead set RAW_SOURCE to the exact CSV/CSV.ZIP path. Do not select an already processed ROA table as the raw input.


In [ ]:
# Uncomment only if numpy or pandas is missing, then restart the kernel.
# %pip install numpy pandas

from pathlib import Path
import numpy as np
import pandas as pd

from pathlib import Path
import sys
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "project_config.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or notebooks directory.")
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
from preparation_config import load_preparation_config
PREP = load_preparation_config(REPO_ROOT)

PROJECT_DIR = PREP["financial_output_root"].parent.parent
OUTPUT_ROOT = PREP["financial_output_root"]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RAW_SOURCE = PREP["compustat_source"]
RAW_FILENAMES = [RAW_SOURCE.name]
RAW_SEARCH_DIRS = [RAW_SOURCE.parent]

BASE_CONTROLS = ["w_roa_current", "w_log_assets", "w_leverage", "w_cash_ratio"]
FE_FIELDS = ["fe_year", "fe_industry_sic2"]
OPTIONAL_CONTROLS = ["w_sales_growth", "w_capx_to_lag_assets", "w_rd_to_assets"]
ROA_GAP_DAYS = (300, 430)          # Broad annual-comparability screen.
STANDARD_ANNUAL_GAP = (350, 380)   # Stricter diagnostic retained for later sample definition.
CODE_VERSION = "compustat_financial_stage1_windows_v1"
print("Outputs:", OUTPUT_ROOT)
print("SEC results and embeddings are NOT needed at this stage.")


## Step 2 — Financial rules and helper functions

Keep INDL (industrial-format), STD (standardized), C (consolidated) records, including active/inactive companies and all downloaded currencies. INDL is a reporting format, not an industry filter; financial firms can still appear. Duplicate GVKEY/datadate rows stop execution and are exported for review. The original fiscal-year label is kept; do not equate it to calendar year.

Current ROA = NI / [(AT + previous AT) / 2]. Require consecutive fiscal years, comparable date gaps, positive current/previous assets, finite inputs and an unchanged nonmissing currency. Compustat apdedate supplies the actual period end when available; datadate is the explicit fallback. The first failed check is recorded in roa_status. Both the broad ROA screen and the stricter normal-year diagnostic remain visible. Dates that differ unusually or map to duplicate effective periods are exported for review.

The next outcome is the immediately next company record's valid ROA, requiring the next consecutive fiscal year, comparable gaps and the same currency. Never skip a missing year to find a later valid outcome. The next accounting period is assumed to start the day after the current effective period end; future SEC timing checks must assess whether that period overlaps disclosure.


In [ ]:
ENGINE_DIGEST = '8ca178266d1308a2d2c331ade36e4e02ee33a687a80ca1db5520fdc0dc4a2726'
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,os,shutil,zipfile,uuid
import numpy as np
import pandas as pd

def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()

def atomic_json(path,value):
    path=Path(path);tmp=path.with_name(path.name+'.'+uuid.uuid4().hex+'.tmp')
    tmp.write_text(json.dumps(value,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
    os.replace(tmp,path)

def save_csv(df,path):
    path=Path(path);tmp=path.with_name(path.name+'.'+uuid.uuid4().hex+'.tmp')
    df.to_csv(tmp,index=False,date_format='%Y-%m-%d',compression='gzip' if path.name.endswith('.gz') else None)
    os.replace(tmp,path)

def clean_cik(s):
    s=s.astype('string').str.strip().str.replace(r'\.0$','',regex=True)
    good=s.str.fullmatch(r'\d{1,10}',na=False)&pd.to_numeric(s,errors='coerce').gt(0)
    return s.where(good).str.zfill(10)

def locate_raw():
    if RAW_SOURCE is not None:
        p=Path(RAW_SOURCE)
        if not p.is_file():raise FileNotFoundError(f'RAW_SOURCE does not exist: {p}')
        return p
    # Search only the configured project folders and Downloads, never the whole drive.
    found=set()
    for folder in RAW_SEARCH_DIRS:
        if not Path(folder).is_dir():continue
        for name in RAW_FILENAMES:
            found.update(p.resolve() for p in Path(folder).rglob(name) if p.is_file())
    paths=sorted(found,key=str)
    if not paths:
        p=input('Paste the full path of the original Compustat CSV or CSV.ZIP: ').strip().strip('"').strip("'")
        path=Path(p)
        if not path.is_file():raise FileNotFoundError(f'File not found: {path}. Set RAW_SOURCE in Step 1.')
        return path
    groups={}
    for p in paths:groups.setdefault(file_hash(p),[]).append(p)
    if len(groups)>1:
        print('Different downloads found:')
        for p in paths:print(p)
        raise ValueError('Set RAW_SOURCE explicitly to the intended original download; no file was selected.')
    print('Found original download:',paths[0])
    return paths[0]

def load_original(path):
    # Read identifiers as strings before any numeric inference.
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:
            names=[n for n in z.namelist() if n.lower().endswith('.csv') and not n.startswith('__MACOSX/')]
            if len(names)!=1:raise ValueError(f'Expected exactly one CSV in ZIP; found {names}')
            with z.open(names[0]) as f:df=pd.read_csv(f,dtype='string',low_memory=False)
    else:df=pd.read_csv(path,dtype='string',low_memory=False)
    df.columns=df.columns.str.strip().str.lower()
    if df.columns.duplicated().any():raise ValueError('Duplicate column names after lowercasing.')
    for c in df.select_dtypes(include='string'):
        df[c]=df[c].str.strip().replace('',pd.NA)
    return df

def failed_status(checks,default):
    return np.select([pd.Series(v).fillna(True).to_numpy(dtype=bool) for v in checks.values()],list(checks),default=default)

def build_financial(raw):
    needed={'gvkey','cik','datadate','fyear','curcd','indfmt','datafmt','consol','at','ni','dlc','dltt','che'}
    if not needed<=set(raw):raise ValueError(f'Missing required raw fields: {sorted(needed-set(raw))}')
    a=raw.loc[raw.indfmt.eq('INDL')&raw.datafmt.eq('STD')&raw.consol.eq('C')].copy()
    if a.empty:raise ValueError('No INDL/STD/C records found.')
    a['gvkey']=a.gvkey.astype('string').str.strip().str.replace(r'\.0$','',regex=True)
    if not a.gvkey.str.fullmatch(r'\d{1,6}',na=False).all():raise ValueError('Missing/invalid GVKEY in filtered data.')
    a['gvkey']=a.gvkey.str.zfill(6)
    original_cik=a.cik.astype('string')
    a['cik']=clean_cik(a.cik)
    a['invalid_nonmissing_cik']=original_cik.notna()&a.cik.isna()
    if 'apdedate' not in a:a['apdedate']=pd.NaT
    for col in ['datadate','apdedate']:a[col]=pd.to_datetime(a[col],errors='raise')
    a['fyear']=pd.to_numeric(a.fyear,errors='raise').astype('Int64')
    if not a[['gvkey','datadate','fyear']].notna().all().all():raise ValueError('Missing company/date/fiscal-year key.')
    a=a.sort_values(['gvkey','datadate']).reset_index(drop=True)
    if a.duplicated(['gvkey','datadate']).any():
        save_csv(a[a.duplicated(['gvkey','datadate'],keep=False)],RUN_DIR/'duplicate_company_dates.csv')
        raise ValueError('Duplicate GVKEY/datadate. Inspect duplicate_company_dates.csv; no arbitrary row selected.')
    for col in ['at','ni','dlc','dltt','che','sale','capx','xrd']:
        a[col]=pd.to_numeric(a[col],errors='raise').astype(float) if col in a else np.nan
    if 'conm' not in a:a['conm']=pd.NA
    a['multiple_periods_in_fyear']=a.duplicated(['gvkey','fyear'],keep=False)
    a['comp_record_id']=a.gvkey+'_'+a.datadate.dt.strftime('%Y%m%d')
    a['effective_end']=a.apdedate.fillna(a.datadate)
    a['period_end_source']=np.where(a.apdedate.notna(),'apdedate','datadate_fallback')
    a['actual_vs_label_gap_days']=(a.effective_end-a.datadate).dt.days
    a['period_end_label_review']=a.actual_vs_label_gap_days.abs().gt(31)
    a['duplicate_effective_period']=a.duplicated(['gvkey','effective_end'],keep=False)
    g=a.groupby('gvkey',sort=False)
    for col in ['datadate','apdedate','effective_end','fyear','curcd','at','sale','multiple_periods_in_fyear']:
        a['previous_'+col]=g[col].shift(1)
    prev_multiple=a.previous_multiple_periods_in_fyear.astype('boolean').fillna(False).astype(bool)
    a['period_gap_days']=(a.datadate-a.previous_datadate).dt.days
    a['actual_gap_days']=(a.effective_end-a.previous_effective_end).dt.days
    checks={
        'no_previous_record':a.previous_datadate.isna(),
        'multiple_periods_in_fyear':a.multiple_periods_in_fyear|prev_multiple,
        'nonconsecutive_fyear':(a.fyear-a.previous_fyear).ne(1),
        'irregular_period_gap':~a.period_gap_days.between(*ROA_GAP_DAYS)|~a.actual_gap_days.between(*ROA_GAP_DAYS),
        'currency_missing_or_changed':a.curcd.isna()|a.previous_curcd.isna()|a.curcd.ne(a.previous_curcd),
        'assets_missing_or_nonpositive':~a['at'].gt(0)|~a.previous_at.gt(0),
        'net_income_missing':a.ni.isna(),
        'nonfinite_financial_values':~np.isfinite(a[['at','previous_at','ni']].to_numpy(dtype=float)).all(axis=1)}
    a['roa_status']=failed_status(checks,'ok')
    a['average_assets']=(a['at']+a.previous_at)/2
    a['roa']=(a.ni/a.average_assets).where(a.roa_status.eq('ok'))
    assets=a['at'].where(a['at'].gt(0)&np.isfinite(a['at']))
    a['w_roa_current']=a.roa
    a['w_log_assets']=np.log(assets)
    a['w_leverage']=(a.dlc+a.dltt)/assets
    a['w_cash_ratio']=a.che/assets
    comparable=((a.fyear-a.previous_fyear).eq(1)&a.period_gap_days.between(*ROA_GAP_DAYS)
       &a.actual_gap_days.between(*ROA_GAP_DAYS)&a.curcd.notna()&a.curcd.eq(a.previous_curcd)
       &~a.multiple_periods_in_fyear&~prev_multiple).fillna(False)
    a['w_sales_growth']=(a.sale/a.previous_sale-1).where(comparable&a.previous_sale.gt(0))
    a['w_capx_to_lag_assets']=(a.capx/a.previous_at).where(comparable&a.previous_at.gt(0))
    a['w_rd_to_assets']=a.xrd/assets
    for c in [c for c in a if c.startswith('w_')]:a[c]=a[c].replace([np.inf,-np.inf],np.nan)
    sic=pd.to_numeric(a['sich'],errors='coerce') if 'sich' in a else pd.Series(np.nan,index=a.index)
    valid=sic.between(100,9999)&sic.eq(np.floor(sic))
    a['sic4']=sic.where(valid).astype('Int64')
    a['sic2']=(a.sic4//100).astype('Int64')
    a['fe_year']=a.fyear.astype('string')
    a['fe_industry_sic2']=a.sic2.astype('string').str.zfill(2)
    a['base_controls_complete']=np.isfinite(a[BASE_CONTROLS].to_numpy(dtype=float)).all(axis=1)
    a['fixed_effect_fields_complete']=a[['fe_year','fe_industry_sic2']].notna().all(axis=1)
    a['current_period_standard_gap']=a.actual_gap_days.between(*STANDARD_ANNUAL_GAP)
    a['negative_debt_or_cash_review']=a[['dlc','dltt','che']].lt(0).any(axis=1)
    g=a.groupby('gvkey',sort=False)
    for col in ['comp_record_id','datadate','apdedate','fyear','curcd','roa','roa_status','effective_end','duplicate_effective_period']:
        a['next_'+col]=g[col].shift(-1)
    a['next_actual_gap_days']=(a.next_effective_end-a.effective_end).dt.days
    future_checks={
        'no_next_record':a.next_comp_record_id.isna(),
        'next_fyear_not_consecutive':(a.next_fyear-a.fyear).ne(1),
        'next_period_gap_irregular':~(a.next_datadate-a.datadate).dt.days.between(*ROA_GAP_DAYS)|~a.next_actual_gap_days.between(*ROA_GAP_DAYS),
        'next_currency_missing_or_changed':a.curcd.isna()|a.next_curcd.isna()|a.curcd.ne(a.next_curcd),
        'next_roa_unavailable':a.next_roa_status.ne('ok')|a.next_roa.isna()}
    a['next_outcome_status']=failed_status(future_checks,'next_fyear_roa_candidate')
    ok=a.next_outcome_status.eq('next_fyear_roa_candidate')
    a['roa_next_fyear']=a.next_roa.where(ok)
    a['next_roa_candidate_standard_gap']=ok&a.next_actual_gap_days.between(*STANDARD_ANNUAL_GAP)
    a['next_period_start']=a.effective_end+pd.Timedelta(days=1)
    a['extreme_current_roa']=a.roa.abs().gt(1)
    a['extreme_future_roa']=a.roa_next_fyear.abs().gt(1)
    a['financial_fields_available']=a.base_controls_complete&a.fixed_effect_fields_complete&a.roa_next_fyear.notna()
    # No SEC timing/identity/sample eligibility is inferred at this stage.
    return a

def run_financial():
    global RUN_DIR
    source=locate_raw();raw_sha=file_hash(source)
    backup_dir=PROJECT_DIR/'raw';backup_dir.mkdir(parents=True,exist_ok=True)
    backup=backup_dir/(raw_sha[:12]+'_'+source.name)
    if backup.exists():
        if file_hash(backup)!=raw_sha:raise ValueError('Raw backup checksum differs.')
    else:
        tmp=backup.with_name(backup.name+'.tmp');shutil.copy2(source,tmp)
        if file_hash(tmp)!=raw_sha:raise ValueError('Raw backup verification failed.')
        os.replace(tmp,backup)
    policy={'version':CODE_VERSION,'code_digest':ENGINE_DIGEST,'raw_sha256':raw_sha,
        'roa_gap_days':ROA_GAP_DAYS,'standard_annual_gap':STANDARD_ANNUAL_GAP,
        'base_controls':BASE_CONTROLS,'fixed_effects':FE_FIELDS}
    runid=hashlib.sha256(json.dumps(policy,sort_keys=True).encode()).hexdigest()[:20]
    RUN_DIR=OUTPUT_ROOT/runid;RUN_DIR.mkdir(parents=True,exist_ok=True)
    receipt_path=RUN_DIR/'financial_snapshot.json'
    if receipt_path.exists():
        receipt=json.loads(receipt_path.read_text(encoding='utf-8'))
        assert receipt['complete'] is True and receipt['policy']==json.loads(json.dumps(policy))
        for info in receipt['files'].values():
            if file_hash(RUN_DIR/info['name'])!=info['sha256']:raise ValueError('Cached output checksum differs; inspect before rerunning.')
        finance=pd.read_csv(RUN_DIR/'financial_panel.csv.gz',dtype={'gvkey':'string','cik':'string','comp_record_id':'string',
            'fe_year':'string','fe_industry_sic2':'string'},low_memory=False)
        for c in [c for c in finance if c.endswith('datadate') or c.endswith('apdedate') or c.endswith('effective_end') or c=='next_period_start']:
            finance[c]=pd.to_datetime(finance[c],errors='raise')
        for c in ['fyear','previous_fyear','next_fyear','sic2','sic4']:
            finance[c]=pd.to_numeric(finance[c],errors='raise').astype('Int64')
        print('Reused verified financial preparation:',RUN_DIR)
        atomic_json(OUTPUT_ROOT/'latest_financial_run.json',{'run_dir':str(RUN_DIR),'snapshot':str(receipt_path)})
        return finance,receipt
    raw=load_original(backup)
    raw_counts={'rows':len(raw),'companies':int(raw.gvkey.nunique()) if 'gvkey' in raw else None,
        'industry_formats':{str(k):int(v) for k,v in raw.indfmt.value_counts(dropna=False).items()} if 'indfmt' in raw else {}}
    finance=build_financial(raw)
    save_csv(finance,RUN_DIR/'financial_panel.csv.gz')
    links=finance[['cik','gvkey']].dropna().drop_duplicates()
    counts=links.groupby('cik').gvkey.nunique().rename('n_candidate_gvkeys')
    links=links.merge(counts,on='cik',validate='many_to_one')
    links['link_status']=np.where(links.n_candidate_gvkeys.eq(1),'single_candidate_not_historically_verified','multiple_candidates_review')
    save_csv(links,RUN_DIR/'cik_gvkey_candidates.csv.gz')
    available=finance.loc[finance.financial_fields_available,
        ['comp_record_id','gvkey','cik','conm','datadate','apdedate','effective_end','fyear','curcd',
         *BASE_CONTROLS,*FE_FIELDS,'next_comp_record_id','next_fyear','next_datadate','next_apdedate',
         'next_effective_end','next_period_start','roa_next_fyear','next_actual_gap_days','next_roa_candidate_standard_gap']]
    save_csv(available,RUN_DIR/'financial_fields_available.csv.gz')
    annual=finance.groupby('fyear',dropna=False).agg(records=('gvkey','size'),companies=('gvkey','nunique'),
        valid_roa=('roa','count'),next_roa_candidates=('roa_next_fyear','count'),
        all_financial_fields=('financial_fields_available','sum')).reset_index()
    save_csv(annual,RUN_DIR/'coverage_by_fyear.csv')
    save_csv(finance.groupby('fe_industry_sic2',dropna=False).agg(records=('gvkey','size'),companies=('gvkey','nunique')).reset_index(),RUN_DIR/'coverage_by_industry.csv')
    potential=finance.roa_next_fyear.notna()
    audit=[]
    for c in BASE_CONTROLS+FE_FIELDS+OPTIONAL_CONTROLS:
        missing=finance[c].isna()
        audit.append({'field':c,'default_control':c in BASE_CONTROLS+FE_FIELDS,
            'missing_all_records':int(missing.sum()),'missing_among_next_roa_candidates':int((missing&potential).sum()),
            'availability_among_next_roa_candidates':float((~missing&potential).sum()/max(1,potential.sum()))})
    save_csv(pd.DataFrame(audit),RUN_DIR/'control_availability.csv')
    save_csv(finance.loc[finance.roa.abs().gt(1)|finance.roa_next_fyear.abs().gt(1),
        ['comp_record_id','gvkey','conm','fyear','curcd','ni','previous_at','at','average_assets','roa','roa_next_fyear']]
        .assign(abs_roa=lambda d:d.roa.abs()).sort_values('abs_roa',ascending=False),RUN_DIR/'extreme_roa_review.csv.gz')
    save_csv(finance.loc[finance.multiple_periods_in_fyear|finance.duplicate_effective_period|finance.period_end_label_review,
        ['comp_record_id','gvkey','conm','fyear','datadate','apdedate','effective_end',
         'multiple_periods_in_fyear','duplicate_effective_period','actual_vs_label_gap_days']],RUN_DIR/'period_review.csv.gz')
    summary={'raw':raw_counts,'financial_rows':len(finance),'companies':int(finance.gvkey.nunique()),
       'valid_roa_records':int(finance.roa.notna().sum()),'next_roa_candidates':int(potential.sum()),
       'all_financial_fields_available':int(finance.financial_fields_available.sum()),
       'missing_cik':int(finance.cik.isna().sum()),'missing_industry':int(finance.fe_industry_sic2.isna().sum()),
       'roa_status':finance.roa_status.value_counts().to_dict(),'next_outcome_status':finance.next_outcome_status.value_counts().to_dict(),
       'currencies':{str(k):int(v) for k,v in finance.curcd.value_counts(dropna=False).items()},
       'next_candidates_outside_standard_gap':int((potential&~finance.next_roa_candidate_standard_gap).sum()),
       'note':'Financial readiness only; no SEC submission timing, report matching, historical identity verification or GENI estimation.'}
    atomic_json(RUN_DIR/'qa_summary.json',summary)
    files={}
    for p in sorted(RUN_DIR.iterdir()):
        if p.is_file() and not p.name.endswith('.tmp'):files[p.name]={'name':p.name,'sha256':file_hash(p)}
    receipt={'complete':True,'stage':'financial_preparation_only','created_at_utc':datetime.now(timezone.utc).isoformat(),
       'policy':policy,'raw_backup':str(backup),'rows':len(finance),'summary':summary,'files':files}
    atomic_json(receipt_path,receipt)
    atomic_json(OUTPUT_ROOT/'latest_financial_run.json',{'run_dir':str(RUN_DIR),'snapshot':str(receipt_path)})
    print('Saved financial preparation:',RUN_DIR)
    return finance,receipt


## Step 3 — Build or reuse the full financial panel

The original filename and content hash identify the backup. Different input bytes or changed code/settings create a new versioned output folder. Completed files are SHA-256 verified before reuse. No existing pilot financial file is overwritten.


In [ ]:
finance, FINANCIAL_SNAPSHOT = run_financial()
print("Raw backup:", FINANCIAL_SNAPSHOT["raw_backup"])
print("Financial records:", len(finance))
print("Companies:", finance["gvkey"].nunique())
print("RUN_DIR:", RUN_DIR)


## Step 4 — Review outcomes, controls and fixed-effect fields

| Field | Calculation / role |
|---|---|
| `roa`, `w_roa_current` | NI divided by average current/previous assets; decimal units |
| `roa_next_fyear` | Valid immediately next fiscal year's ROA; proposed outcome Y |
| `w_log_assets` | Natural logarithm of positive total assets, in the downloaded currency's units |
| `w_leverage` | (DLC + DLTT) / AT; missing debt components stay missing |
| `w_cash_ratio` | CHE / AT |
| `fe_year` | Current Compustat fiscal-year category |
| `fe_industry_sic2` | Two-digit historical SIC category, from `sich`; missing stays missing |
| `w_sales_growth` | SALE / previous SALE − 1, requiring a comparable prior year and positive previous sales |
| `w_capx_to_lag_assets` | CAPX / previous AT, requiring a comparable prior year and positive previous assets |
| `w_rd_to_assets` | XRD / current AT; missing XRD is never assumed zero |

The default model specification remains **four financial W + year and industry categories**. Preparing these categories does not itself fit fixed effects. The later GENI nuisance models must one-hot encode them within training folds; do not treat industry codes as a continuous numeric variable. Use the same controls/folds across embedding comparisons. Firm fixed effects are not added here.

Optional controls are calculated for assessment but not automatically added to W. Sales growth may be a useful additional specification if available and temporally justified. CAPX and R&D may reflect channels affected by competition; do not add them mechanically. Financial availability is not proof of a valid pretreatment adjustment set.

USD/CAD ratios are unitless within records, but log assets are not currency-harmonized. Keep `curcd` for a later common-currency restriction or currency-control decision; do not mix raw log assets across currencies without addressing this. Retain extreme ROA for review rather than deleting based on its sign or magnitude.


In [ ]:
summary = FINANCIAL_SNAPSHOT["summary"]
print("Valid current ROA:", summary["valid_roa_records"])
print("Next-year ROA candidates:", summary["next_roa_candidates"])
print("Outcome + four W + year/industry available:", summary["all_financial_fields_available"])
print("Missing CIK:", summary["missing_cik"], "| Missing SIC2:", summary["missing_industry"])
print("Next candidates outside normal-year gap:", summary["next_candidates_outside_standard_gap"])
print("\nCurrent ROA status:")
print(finance["roa_status"].value_counts())
print("\nCurrency counts:")
print(finance["curcd"].value_counts(dropna=False))
print("\nROA distribution — decimal units (0.05 = 5%):")
display(finance[["roa", "roa_next_fyear"]].describe(percentiles=[.01,.5,.99]))
print("\nControl availability:")
display(pd.read_csv(RUN_DIR / "control_availability.csv"))
print("\nAnnual coverage:")
display(pd.read_csv(RUN_DIR / "coverage_by_fyear.csv"))
display(finance[["comp_record_id","gvkey","conm","cik","fyear","datadate","apdedate",
    *BASE_CONTROLS,*FE_FIELDS,"roa_next_fyear"]].head(10))


## Step 5 — Verify outputs and hand off to the later SEC/report join

| Output | Use |
|---|---|
| `financial_panel.csv.gz` | Full cleaned company-period panel, all outcomes, controls and audit flags |
| `financial_fields_available.csv.gz` | Financial-field-available subset only; **not the final eligible sample** |
| `cik_gvkey_candidates.csv.gz` | All downloaded CIK–GVKEY candidates; a unique candidate is not historical identity certification |
| `control_availability.csv` | Missingness of default and optional control fields |
| `coverage_by_fyear.csv`, `coverage_by_industry.csv` | Financial coverage; not report-match rates |
| `period_review.csv.gz`, `extreme_roa_review.csv.gz` | Date/period and extreme-value audit lists |
| `qa_summary.json`, `financial_snapshot.json` | Counts, policy, hashes and completion receipt |

**After SEC finishes:** use `report_filing_dates_all.csv.gz` + this full financial panel + MiniLM's paired report index/matrix. First establish the CIK–GVKEY candidate, resolve historical/ambiguous identities as needed, and match the SEC reporting-period end to the financial effective end. Preserve unresolved reports in an audit table. Review duplicate submissions/company-period reports, normal period lengths and the timing of the future outcome relative to filing. Join the embedding by `doc_id`, never by CSV position or fiscal year alone.

There is no need to re-encode MiniLM or read the 1,238 Excel parts. All three embedding models can reuse this financial preparation. The next stage will write a new aligned report-level table/matrix pair after these checks. This notebook stops here and does not claim causal identification or GENI readiness.


In [ ]:
assert FINANCIAL_SNAPSHOT["complete"] is True
assert finance["comp_record_id"].is_unique
for item in FINANCIAL_SNAPSHOT["files"].values():
    assert file_hash(RUN_DIR / item["name"]) == item["sha256"], item["name"]
assert not finance.loc[finance["roa_status"].ne("ok"), "roa"].notna().any()
assert not finance.loc[finance["next_outcome_status"].ne("next_fyear_roa_candidate"), "roa_next_fyear"].notna().any()
print("FINANCIAL PREPARATION COMPLETE — verified.")
print("Full panel:", RUN_DIR / "financial_panel.csv.gz")
print("Summary:", RUN_DIR / "qa_summary.json")
print("This is financial preparation, not the final SEC/report/embedding match.")
